## Entrenamiento experimental de personas · módulo INTENCIONES

Este cuaderno se abre desde **GitHub** y entrena YOLO11n Detect con **GPU de Google Colab**. No cambia el modelo vial ni el sistema de seguimiento actual.

**Antes de ejecutarlo:** en Colab selecciona `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`, y prepara imágenes y etiquetas YOLO de **personas** en Drive. BEHAVE, KTH y UT-Interaction de `datos_intenciones/evaluacion` quedan **reservados para evaluación**, no para entrenamiento. El entrenamiento no es válido si solo tienes videos sin etiquetas.

Estructura esperada en Google Drive:

```text
Mi unidad/detector_cascos_motos/datasets/personas_v1/
  images/train/*.jpg
  images/val/*.jpg
  labels/train/*.txt
  labels/val/*.txt
```

Cada línea de etiqueta debe ser `0 x_centro y_centro ancho alto` con coordenadas normalizadas entre 0 y 1. Solo se admite la clase `0: person`. Un `.txt` ausente o vacío es válido para una imagen sin personas. Separar por video/cámara antes de extraer fotogramas para evitar fugas entre train y val.

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "pyyaml>=6"

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No se detectó GPU CUDA. Activa GPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución antes de entrenar.")
print("GPU disponible:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import subprocess
REPO_URL = "https://github.com/eduardogrados644-lang/detector_cascos_motos.git"
RAMA = "mejoras-modulos-vial-conductual"
REPO = Path("/content/detector_cascos_motos")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", RAMA], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", RAMA], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", RAMA], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA, REPO_URL, str(REPO)], check=True)
GIT_SHA = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Repositorio sincronizado:", GIT_SHA)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_BASE = Path("/content/drive/MyDrive/detector_cascos_motos")
DATASET = DRIVE_BASE / "datasets" / "personas_v1"
SALIDAS = DRIVE_BASE / "modelos" / "personas"
if not DATASET.is_dir():
    raise FileNotFoundError(f"Falta el dataset YOLO etiquetado en Drive: {DATASET}")
if "evaluacion" in [p.lower() for p in DATASET.parts]:
    raise ValueError("No entrenar con datasets reservados para evaluación")
print("Carpeta de entrenamiento:", DATASET)
print("Resultados en Drive:", SALIDAS)

In [ ]:
from collections import Counter
import yaml
EXTS = {".jpg", ".jpeg", ".png", ".webp"}
conteos = {}
nombres_por_split = {}
for split in ("train", "val"):
    imagenes_dir, etiquetas_dir = DATASET / "images" / split, DATASET / "labels" / split
    if not imagenes_dir.is_dir() or not etiquetas_dir.is_dir():
        raise FileNotFoundError(f"Faltan carpetas images/{split} o labels/{split} dentro de {DATASET}")
    imagenes = sorted(p for p in imagenes_dir.rglob("*") if p.is_file() and p.suffix.lower() in EXTS)
    if not imagenes:
        raise ValueError(f"No hay imágenes en images/{split}")
    nombres_por_split[split] = {p.name.lower() for p in imagenes}
    instancias = 0
    for imagen in imagenes:
        etiqueta = etiquetas_dir / (imagen.stem + ".txt")
        if not etiqueta.exists():
            continue  # Negativo sin personas; YOLO acepta etiqueta ausente.
        for linea in etiqueta.read_text(encoding="utf-8-sig").splitlines():
            if not linea.strip():
                continue
            partes = linea.split()
            if len(partes) != 5 or partes[0] != "0":
                raise ValueError(f"Etiqueta incorrecta (solo clase persona 0): {etiqueta}: {linea}")
            x, y, w, h = (float(n) for n in partes[1:])
            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1 and x-w/2 >= -1e-5 and y-h/2 >= -1e-5 and x+w/2 <= 1.00001 and y+h/2 <= 1.00001):
                raise ValueError(f"Coordenadas YOLO inválidas: {etiqueta}: {linea}")
            instancias += 1
    if instancias == 0:
        raise ValueError(f"No hay etiquetas positivas de personas en {split}")
    conteos[split] = {"imagenes": len(imagenes), "personas_etiquetadas": instancias}
if nombres_por_split["train"] & nombres_por_split["val"]:
    raise ValueError("Hay archivos con el mismo nombre en train y val; verifica que ningún video/imagen esté en ambos conjuntos")
# Las imágenes de una misma escena/video deben estar en un único split; esto requiere revisión humana adicional.
DATA_YAML = Path("/content/personas_v1.yaml")
DATA_YAML.write_text(yaml.safe_dump({"path": str(DATASET), "train": "images/train", "val": "images/val", "names": {0: "person"}}, sort_keys=False, allow_unicode=True), encoding="utf-8")
print("Datos validados:", conteos)
print("Configuración:", DATA_YAML)

In [ ]:
from datetime import datetime, timezone
from ultralytics import YOLO
SESION = datetime.now(timezone.utc).strftime("personas_%Y%m%d_%H%M%S_utc")
# yolo11n.pt es un modelo PREENTRENADO; solo se ajusta a los datos etiquetados.
modelo = YOLO("yolo11n.pt")
modelo.train(
    data=str(DATA_YAML),
    epochs=25,
    patience=7,
    imgsz=640,
    batch=-1,            # Ultralytics ajusta el lote a la memoria de la GPU.
    device=0,
    workers=2,
    cache=False,
    project="/content/entrenamientos_intenciones",
    name=SESION,
    exist_ok=False,
    seed=42,
    plots=True,
)
CARPETA_RESULTADOS = Path(modelo.trainer.save_dir)
BEST = CARPETA_RESULTADOS / "weights" / "best.pt"
LAST = CARPETA_RESULTADOS / "weights" / "last.pt"
if not BEST.is_file():
    raise FileNotFoundError(f"El entrenamiento no generó best.pt: {BEST}")
print("Pesos entrenados:", BEST)

In [ ]:
from ultralytics import YOLO
modelo_validado = YOLO(str(BEST))
metricas = modelo_validado.val(data=str(DATA_YAML), imgsz=640, device=0, plots=True)
print("mAP50:", metricas.box.map50, "mAP50-95:", metricas.box.map)
print("No es una prueba externa: la evaluación definitiva se hará en videos reservados y etiquetados.")

In [ ]:
import json
import shutil
SALIDA = SALIDAS / SESION
SALIDA.mkdir(parents=True, exist_ok=False)
shutil.copy2(BEST, SALIDA / "best.pt")
if LAST.is_file():
    shutil.copy2(LAST, SALIDA / "last.pt")
resumen = {"sesion": SESION, "rama": RAMA, "git_sha": GIT_SHA, "modelo_base": "yolo11n.pt", "clase": {"0": "person"}, "dataset": str(DATASET), "conteos": conteos, "epocas_objetivo": 25, "imgsz": 640, "mAP50_val": float(metricas.box.map50), "mAP50_95_val": float(metricas.box.map)}
(SALIDA / "resumen.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False), encoding="utf-8")
print("Modelo y métricas guardados en Drive:", SALIDA)
print("Para usarlo en Windows: descarga best.pt y cópialo a modelos_intenciones/personas/best.pt")

### Integración posterior en VS Code

Después de descargar `best.pt`, **no** reemplaces `yolo11n-pose.pt` por este peso: son tareas diferentes (Detect vs. Pose). Primero compara el detector entrenado con el detector de personas original usando `yolo detect predict ... model=modelos_intenciones/personas/best.pt classes=0`, y mide el seguimiento y la cantidad de personas visibles. Solo tras validar la mejora se integrará con el módulo conductual.

**Seguridad y reproducibilidad:** no subas videos privados, datos personales, credenciales, tokens ni pesos sin verificar licencia a GitHub; este cuaderno solo descarga código y guarda el modelo en **tu Drive**. Los nombres de archivos coincidentes no son una prueba completa de fuga de datos: separar también por video, personas/escena y fecha. La selección final se evalúa con videos de referencia no empleados durante el entrenamiento.